# Закупки группы Сбер за 2024–2025 годы

Основной аналитический отчёт в формате Jupyter Notebook. Внутри есть описание источников, обработки, базы данных, SQL-запросов, аналитического модуля, LLM-обработки и визуализаций.

In [1]:
import json
from pathlib import Path
import pandas as pd
from IPython.display import SVG, display
root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
stage2 = root / 'data' / 'processed' / 'stage2'
stage3 = root / 'data' / 'processed' / 'stage3'
stage4 = root / 'data' / 'processed' / 'stage4'
charts = stage4 / 'charts'


## Что сделано

Проект собирает закупки юридических лиц группы Сбер из открытых источников, связывает пересекающиеся записи, обезличивает персональные данные, очищает дубли, загружает аналитический слой в PostgreSQL и строит исследование по ключевому направлению — IT и телеком.

В итоговой выборке 1 458 канонических закупок за 2024–2025 годы. Для части процедур есть сумма, дата, заказчик, источник, статус и ссылка на карточку.

## Источники и сбор

| Источник | Зачем использован |
|---|---|
| ЕИС zakupki.gov.ru | Федеральный открытый источник, полезен для 44-ФЗ/223-ФЗ и проверки отдельных карточек |
| Сбербанк-АСТ / SberB2B | Основной массив корпоративных процедур группы Сбер |
| Росэлторг, ЗаказРФ, ЛотОнлайн, ТекТорг, РТС-тендер, ЕТП ГПБ | Проверялись как дополнительные открытые площадки и источники ссылок |
| ЦБ РФ | Внешние факторы: USD/RUB и ключевая ставка |

Сбор сделан скриптами этапа 1. После выгрузки данные приводятся к единой структуре, источники связываются через номер закупки, ссылку, заказчика, дату и сумму. Персональные данные в экспортных слоях маскируются.

## Обработка и база данных PostgreSQL

Аналитическая схема `sber_procurement` содержит таблицы:

| Таблица | Назначение |
|---|---|
| `organizations` | юридические лица группы Сбер |
| `purchases` | каноническая закупочная процедура после дедупликации |
| `purchase_sources` | представления закупки в разных источниках |
| `purchase_candidates` | широкие поисковые кандидаты для контроля полноты |
| `duplicate_audit` | аудит найденных дублей |
| `documents` | очередь карточек и документов для дальнейшего парсинга |

Основная связь дочерних таблиц с закупками идёт через `canonical_purchase_id`. Для аналитики подготовлены SQL-запросы в `stages/stage2/sql/analytics_queries.sql`: динамика по месяцам, топ заказчиков, топ лотов, проверка дублей и закупки из нескольких источников.

## Ключевое направление анализа

Выбрано направление IT и телеком: 210 закупок на 906,8 млн руб., то есть 14,4% подтверждённой выборки по количеству. Направление подходит для проверки гипотезы о связи с USD/RUB, потому что в нём могут быть оборудование, лицензии, ПО и телеком-компоненты.

## Сравнение 2024 и 2025

В 2024 году найдено 325 закупок на 1 363,9 млн руб.; в 2025 году — 1 133 закупки на 2 912,6 млн руб. Рост есть и по количеству, и по сумме, но он интерпретируется осторожно: часть эффекта связана с покрытием источников и отличиями в полноте сумм.

## Корреляционный анализ и гипотезы

Проверены четыре гипотезы по 24 месячным наблюдениям: сумма и количество IT-закупок против USD/RUB, сумма и количество строительных закупок против ключевой ставки. Для сумм использовалось преобразование `log1p`, значимость проверялась на уровне 0,05.

Результат: связь суммы IT-закупок с USD/RUB не подтверждена; связь количества IT-закупок с USD/RUB получилась отрицательной и статистически значимой; связь строительных закупок с ключевой ставкой не подтверждена.

## Аномалии

Сформированы флаги для ручной проверки: экстремальные суммы, отменённые процедуры, повторные публикации и скачки цены по похожему предмету. Эти флаги не доказывают нарушение — они показывают, какие карточки и документы стоит открыть в первую очередь.

## Что дала LLM-обработка

Локальная Qwen через Ollama обработала первую часть приоритетной очереди карточек. Результат — структурированный JSON и черновики выводов в формате `наблюдение / интерпретация / значимость / ограничение`. Все 38 обработанных ответов были валидным JSON.

Главная польза LLM — автоматизация работы с короткими неструктурированными описаниями и ускорение подготовки аналитических карточек. Важное ограничение: модель не считается источником фактов о победителях, единственном участнике или нарушениях. Если в данных нет протоколов, такие выводы требуют ручной проверки.

## Динамика объёма закупок по месяцам

**Наблюдение:** Пик по количеству процедур пришёлся на 2025-11 (181 закупок), а максимальная указанная сумма — на 2025-02 (413.6 млн руб.). В 2025 году публикаций заметно больше, чем в 2024.

**Интерпретация:** Разрыв между пиком количества и пиком суммы показывает, что динамику формируют два разных эффекта: массовые небольшие публикации и единичные крупные лоты. Рост 2025 года нельзя автоматически трактовать как чистый рост закупочной активности: часть эффекта связана с расширением сбора и лучшим покрытием источников.

**Значимость:** Для закупочного анализа важно смотреть одновременно на количество, сумму и медиану. Иначе один крупный лот может создать ощущение общего роста, а поток небольших процедур — завысить оценку операционной активности.

**Ограничение:** Используется дата публикации, а не дата оплаты или исполнения. Итоговая цена контракта в текущем слое данных отсутствует, поэтому анализ построен по указанной начальной сумме.

![Динамика объёма закупок по месяцам](charts/01_monthly_dynamics.svg)

Файл графика в репозитории: `data/processed/stage4/charts/01_monthly_dynamics.svg`

## Структура закупок по направлениям

**Наблюдение:** IT и телеком выросли с 61 до 149 процедур, а указанная сумма увеличилась на 254.88%. Логистика и эксплуатация дали самый резкий прирост по количеству: с 13 до 119. Строительство выросло по числу процедур, но сумма снизилась на -5.24%.

**Интерпретация:** Направления ведут себя неодинаково: в IT рост сопровождается ростом суммы, в строительстве рост количества может означать дробление работ или большее число малых процедур, а не рост бюджета. Категория `other` остаётся крупной, потому что часть предметов закупки описана общими формулировками.

**Значимость:** Выбор ключевого направления для исследования обоснован: IT и телеком достаточно велики по числу и сумме, а также потенциально чувствительны к валютному фактору из-за оборудования, лицензий и импортных компонентов.

**Ограничение:** Классификация сделана по ключевым словам в названии закупки. Для точной отраслевой структуры нужны ОКПД2, технические задания и нормализация предметов закупки.

![Структура закупок по направлениям](charts/02_category_structure.svg)

Файл графика в репозитории: `data/processed/stage4/charts/02_category_structure.svg`

## Топ-20 самых дорогих лотов

**Наблюдение:** Самая крупная процедура — SBR028-2502050031.1 на 296.1 млн руб. Топ-20 заметно выделяется относительно основной массы закупок.

**Интерпретация:** Агрегированная сумма чувствительна к небольшому числу крупных процедур. Это особенно важно для сравнения годов: изменение одного-двух лотов может выглядеть как изменение всего направления.

**Значимость:** Топ-20 нужен как контроль выбросов. Перед управленческим выводом полезно проверять, не держится ли рост суммы на отдельных нетипичных процедурах.

**Ограничение:** Поле суммы может означать НМЦ, лимит, тариф или единичную ставку. Без документации и итогового контракта нельзя утверждать фактический расход.

![Топ-20 самых дорогих лотов](charts/03_top20_lots.svg)

Файл графика в репозитории: `data/processed/stage4/charts/03_top20_lots.svg`

## Корреляция: IT-закупки и USD/RUB

**Наблюдение:** Связь суммы IT-закупок с USD/RUB статистически не подтверждена: r=-0.1039, p=0.6501. Для количества IT-процедур получена отрицательная статистически значимая связь: r=-0.4957, p=0.0135.

**Интерпретация:** Гипотеза о прямом синхронном росте суммы IT-закупок при росте доллара в этих данных не подтверждается. Отрицательная связь по количеству может отражать сезонность, лаг между планированием и публикацией, смещение состава источников или то, что закупки заранее планируются бюджетными циклами.

**Значимость:** Результат защищает от слишком простого вывода `доллар вырос — IT-закупки выросли`. В текущей выборке валютный фактор нужно проверять с лагами и детализацией по оборудованию/ПО.

**Ограничение:** Всего 24 месячных наблюдения. Не проверены лаги, сезонность, типы IT-закупок и доля импортной компоненты.

![Корреляция: IT-закупки и USD/RUB](charts/04_it_vs_usd.svg)

Файл графика в репозитории: `data/processed/stage4/charts/04_it_vs_usd.svg`

## Корреляция: строительство и ключевая ставка

**Наблюдение:** Связь суммы строительных закупок с ключевой ставкой незначима: r=0.0134, p=0.9510.

**Интерпретация:** В пределах 2024–2025 годов закупки строительства, ремонта и эксплуатации сильнее зависят от проектных циклов, бюджета и конкретных объектов, чем от синхронного уровня ставки в месяце публикации.

**Значимость:** Гипотеза о ставке не подтверждена текущим агрегатом, поэтому её нельзя использовать как объяснение динамики без дополнительных факторов.

**Ограничение:** Нужны более длинный ряд, лаги, региональность, тип работ, срок исполнения и стоимость финансирования конкретных проектов.

![Корреляция: строительство и ключевая ставка](charts/05_construction_vs_key_rate.svg)

Файл графика в репозитории: `data/processed/stage4/charts/05_construction_vs_key_rate.svg`

## Распределение суммы закупок

**Наблюдение:** Сумма отсутствует у 278 процедур. Остальные закупки распределены от символических/малых значений до лотов свыше 100 млн руб.

**Интерпретация:** В одном поле смешаны разные экономические смыслы: НМЦ, тариф, единичная цена, лимит или пустое значение. Поэтому средняя сумма без сегментации и фильтров может вводить в заблуждение.

**Значимость:** Перед расчётом средних, темпов роста и корреляций надо явно фиксировать, что анализируется: полная цена лота или только опубликованное числовое поле карточки.

**Ограничение:** Для нормализации нужны тип цены, единица измерения, объём поставки и документы закупки.

![Распределение суммы закупок](charts/06_amount_distribution.svg)

Файл графика в репозитории: `data/processed/stage4/charts/06_amount_distribution.svg`

## НМЦ и итоговая цена контракта

**Наблюдение:** Начальная сумма доступна у 1180 процедур из 1458 фактических строк, итоговая цена контракта в текущем слое — у 0.

**Интерпретация:** Поисковые карточки и выгрузки площадок дают хороший слой для анализа публикаций, но не закрывают контрактный слой. Поэтому экономию НМЦ → контракт пока корректно посчитать нельзя.

**Значимость:** Это честное ограничение проекта: в отчёте показано, где данные уже пригодны для анализа, а где требуется дополнительное извлечение протоколов и контрактов.

**Ограничение:** Нужно скачать и распарсить протоколы, сведения о победителе, количестве участников и итоговой цене.

![НМЦ и итоговая цена контракта](charts/07_nmc_contract_availability.svg)

Файл графика в репозитории: `data/processed/stage4/charts/07_nmc_contract_availability.svg`

## Результат обработки через LLM

**Наблюдение:** Локальная Qwen через Ollama обработала 38 карточек, все ответы были валидным JSON (100.0%). Покрытие очереди LLM — 8.41%. 26 ответов отмечены как требующие ручной проверки формулировок.

**Интерпретация:** LLM хорошо справилась со структурированием коротких описаний: привела выводы к единому формату, помогла приоритизировать карточки и сформировать черновики интерпретаций. Но модель иногда использует рискованные слова про поставщика, нарушения или единственного участника там, где во входных данных нет протоколов.

**Значимость:** Практический результат LLM — не `автоматическое доказательство аномалий`, а ускорение аналитической рутины: черновая разметка, объяснение риска и список карточек для ручной проверки документов.

**Ограничение:** Обработана только первая часть приоритетной очереди, выборка не случайная. Для финальных выводов по конкуренции нужны протоколы участников, победители и контрактные цены.

![Результат обработки через LLM](charts/08_llm_quality.svg)

Файл графика в репозитории: `data/processed/stage4/charts/08_llm_quality.svg`

## Итоговый вывод

Проект даёт воспроизводимый аналитический слой по закупкам группы Сбер за 2024–2025 годы. Основной рост виден в 2025 году, но он зависит от качества покрытия источников, наличия сумм и отдельных крупных процедур. IT и телеком — наиболее содержательное направление для углублённого анализа, однако простая гипотеза о прямой связи суммы IT-закупок с курсом USD/RUB не подтвердилась.

Для следующего шага важнее всего извлечь документы процедур: протоколы, победителей, количество участников и итоговые цены контрактов. Тогда можно будет проверять экономию, конкуренцию и доли поставщиков уже не как гипотезы, а как полноценные метрики.

## Как демонстрировать Notebook

1. Открыть `notebooks/final_sber_procurement_report.ipynb`.
2. В PyCharm или Jupyter нажать `Restart Kernel and Run All Cells`.
3. Показать сверху вниз: источники, обработку, схему БД, SQL, аналитику, LLM и графики.
4. Графики встроены в Notebook как attachments, поэтому должны быть видны сразу после открытия без доверия к code outputs.